In [41]:
from transformers import pipeline

pipe = pipeline("text-generation", model="openai-community/gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
prompt="I am Roshan Bhaskar"
print(pipe(prompt))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'I am Roshan Bhaskar. I am an old man. I have become an Indian. I am a member of the Kalkand-based Muslim-led movement and I am the person of record in this country. I am a member of the National Security Council that is now the government of India. I am a member of the Executive Council of the Council of Ministers and I am a member of the Council of the Federal Reserve Bank of New York. I am a member of the Supreme Court of the United States of America and I am a member of the Council of the Federal Reserve Board. I am a member of the United Nations Security Council. I am the third member of the Supreme Court of the United States, the last one to be confirmed by the Senate. I am the first woman to serve as a member of the Supreme Court of the United States. I am a member of the Constitution Committee of the Senate. I am the third member of the Senate Constitution Committee. I am the first Muslim to serve as a member of the Supreme Court of the United States. I am t

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

In [45]:
sentence="Unsure"
input_ids=tokenizer(sentence,return_tensors="pt")["input_ids"]

In [46]:
input_ids

tensor([[ 3118, 19532]])

In [47]:
for token_id in input_ids[0]:
  print(tokenizer.decode(token_id))

Un
sure


In [48]:
sentence="I like machine learning because"
token_ids=tokenizer(sentence,return_tensors="pt").input_ids

In [49]:
token_ids

tensor([[  40,  588, 4572, 4673,  780]])

In [50]:
outputs=model(token_ids).logits[0,-1]

In [51]:
outputs.argmax()

tensor(340)

In [52]:
tokenizer.decode(outputs.argmax())

' it'

In [53]:
import torch

In [54]:
sentence="I like machine learning to enhance,"
token_ids=tokenizer(sentence,return_tensors="pt").input_ids
outputs=model(token_ids).logits[0,-1]
final_logits=torch.topk(outputs,20)
for index in final_logits.indices:
  print(tokenizer.decode(index))

 not
 but
 and
 I
 rather
 because
 so
 for
 to
 or
 improve
 as
 especially
 it
 in
 which
 while
 even
 we
 enhance


In [55]:
def greedy_decode(logits):
  return torch.argmax(logits)
tokenizer.decode(greedy_decode(outputs))

' not'

In [56]:
torch.softmax(final_logits.values,dim=0)

tensor([0.3076, 0.2850, 0.1229, 0.0338, 0.0303, 0.0283, 0.0260, 0.0214, 0.0211,
        0.0210, 0.0172, 0.0165, 0.0098, 0.0095, 0.0089, 0.0089, 0.0083, 0.0080,
        0.0078, 0.0077], grad_fn=<SoftmaxBackward0>)

In [57]:
import torch.nn as nn

In [58]:
def top_k(logits,k):
  torch.topk(logits)
def top_k_sampling(logits,k=50):
  values,indices=torch.topk(logits,k)
  probs=torch.softmax(values,dim=-1)
  sampled=torch.multinomial(probs,1)
  return indices[sampled]

In [59]:
tokenizer.decode(top_k_sampling(outputs))

' not'

In [60]:
def top_p_sampling(logits,p=0.9):
  sorted_logits,sorted_indices=torch.sort(logits,descending=True)
  sorted_probs=torch.softmax(sorted_logits,dim=-1)
  cummulative_probs=sorted_probs.cumsum(dim=-1)
  mask=cummulative_probs>p
  sorted_logits[mask]=float("-inf")
  filtered_probs=torch.softmax(sorted_logits,dim=-1)
  sampled=torch.multinomial(filtered_probs,1)
  return sorted_indices[sampled]

In [61]:
outputs

tensor([-125.2748, -125.6782, -128.3346,  ..., -128.2601, -132.3935,
        -123.7939], grad_fn=<SelectBackward0>)

In [62]:
top_p_sampling(outputs,0.9)

tensor([407])

In [63]:
tokenizer.decode(top_p_sampling(outputs,0.9))

' not'

In [64]:
def temperature_sampling(logits,temperature=1.0):
  scaled=logits/temperature
  probs=torch.softmax(scaled,dim=-1)
  return torch.multinomial(probs,1)

In [65]:
temperature_sampling(outputs,temperature=1)

tensor([475])

In [66]:
tokenizer.decode(temperature_sampling(outputs,temperature=1))

' eventually'

In [67]:
def random_sampling(logits):
  probs=torch.softmax(logits,dim=-1)
  return torch.multinomial(probs,1)

In [68]:
tokenizer.decode(random_sampling(outputs))

' not'

In [69]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

In [70]:
import pandas as pd

In [71]:
(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [72]:
import pandas as pd

In [73]:
ds["train"].to_pandas()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [74]:
from transformers import pipeline

In [75]:
classifier=pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [76]:
def score(example):
    return {
        "model_prediction": classifier(example["text"][:500])[0]["label"]
    }

ds["train"] = ds["train"].map(score)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [77]:
from datasets import load_dataset

ds = load_dataset("ucirvine/sms_spam")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

In [78]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})


In [79]:
ds["train"].to_pandas()

,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...
5569,This is the 2nd time we have tried 2 contact u...,1
5570,Will ü b going to esplanade fr home?\n,0
5571,"Pity, * was in mood for that. So...any other s...",0
5572,The guy did some bitching but I acted like i'd...,0


In [80]:
df = ds["train"].to_pandas()

df["sms"] = df["sms"].str.replace("\n", " ")
df["sms"] = df["sms"].str.strip()

In [81]:
df.duplicated().sum()

np.int64(414)

In [82]:
df = df.drop_duplicates()

In [83]:
df["label"].value_counts()

,count
label,
0,4518
1,642


In [84]:
split = ds["train"].train_test_split(test_size=0.2)
train_df = split["train"]
test_df = split["test"]

In [85]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [86]:
def tokenize(example):
    return tokenizer(
        example["sms"],
        truncation=True,
        max_length=128
    )

train_ds = train_df.map(tokenize, batched=True)
test_ds = test_df.map(tokenize, batched=True)

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [87]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [88]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [89]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

In [90]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    save_strategy="epoch"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [91]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [92]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,  # 🔥 THIS enables dynamic padding
    compute_metrics=compute_metrics
)

In [93]:
trainer.train()

Step,Training Loss
500,0.054076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=837, training_loss=0.036439294883427224, metrics={'train_runtime': 108.7106, 'train_samples_per_second': 123.051, 'train_steps_per_second': 7.699, 'total_flos': 214338460434720.0, 'train_loss': 0.036439294883427224, 'epoch': 3.0})

In [94]:
trainer.evaluate()

{'eval_loss': 0.034177377820014954,
 'eval_accuracy': 0.9919282511210762,
 'eval_f1': 0.9644268774703557,
 'eval_precision': 0.976,
 'eval_recall': 0.953125,
 'eval_runtime': 2.5694,
 'eval_samples_per_second': 433.957,
 'eval_steps_per_second': 27.244,
 'epoch': 3.0}